# 협업용 파생 피처 표준 예제

이 노트북은 피처 아이디어를 검토하고 결과를 눈으로 확인하는 용도입니다. 실제 모델 파이프라인의 단일 기준은 `feature_example.py`입니다. 노트북에만 존재하는 피처 로직을 만들지 않습니다.

공통 계약은 다음과 같습니다.

- 입력 한 행 → 출력 한 행
- `row_id` 값과 순서 보존
- 출력은 `row_id + 새 피처`만 포함
- 모듈 고유 prefix 사용
- 현재 타깃, `row_id` 숫자, 다른 test 행의 통계 사용 금지
- 학습이 필요한 피처는 사전 생성하지 않고 CV fold 내부에서 fit/transform


## 1. 공용 모듈 불러오기

각 팀원은 `feature_example.py`를 복사한 뒤 `SPEC`, `PARAMS`, `build_features()`만 자신의 피처에 맞게 수정합니다. 검증·조립·manifest 코드는 유지합니다.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

HERE = Path.cwd().resolve()
if not (HERE / 'feature_example.py').exists():
    raise FileNotFoundError('노트북을 control_success_feature_template 폴더에서 실행하세요.')
PROJECT_ROOT = HERE.parents[1]
sys.path.insert(0, str(HERE))

from feature_example import (
    ID_COLUMN,
    SPEC,
    PARAMS,
    assert_row_independence,
    build_features,
    feature_summary,
    merge_feature_blocks,
    validate_feature_block,
)

print(json.dumps({
    'name': SPEC.name,
    'version': SPEC.version,
    'owner': SPEC.owner,
    'prefix': SPEC.prefix,
    'feature_count': len(SPEC.feature_columns),
    'stateful': SPEC.stateful,
}, ensure_ascii=False, indent=2))

{
  "name": "demo_situation_history",
  "version": "1.0.0",
  "owner": "CHANGE_ME",
  "prefix": "demo__",
  "feature_count": 25,
  "stateful": false
}


## 2. 필요한 컬럼만 읽기

전체 CSV의 모든 컬럼을 읽지 않고 `SPEC.required_columns`만 로드합니다. 개발 중에는 20,000행 스모크 테스트로 확인하고, 통과한 뒤 전체 데이터를 실행합니다.

In [2]:
NROWS = 20_000
read_kwargs = {
    'usecols': list(SPEC.required_columns),
    'low_memory': False,
}
train = pd.read_csv(PROJECT_ROOT / 'data' / 'train.csv', nrows=NROWS, **read_kwargs)
test = pd.read_csv(PROJECT_ROOT / 'data' / 'test.csv', **read_kwargs)

print('train:', train.shape, 'test sample:', test.shape)
train.head(3)

train: (20000, 19) test sample: (5, 19)


,row_id,inning,balls_before,strikes_before,score_diff_pitcher_team,runner_on_1b,runner_on_2b,runner_on_3b,li,pitcher_hand,batter_hand,asof_pitcher_n,asof_pitcher_success_rate,asof_pitcher_reverse_rate,asof_pitcher_middle_rate,asof_pitcher_prev3_game_success_rate,asof_pitcher_prev5_game_success_rate,asof_batter_n,asof_batter_success_rate
0,TRAIN_0000001,1,0,0,0,0,0,0,0.87,1,2,0,NaN,NaN,NaN,NaN,NaN,0,NaN
1,TRAIN_0000002,1,0,0,0,1,0,0,1.44,1,1,1,0.0,1.0,0.0,NaN,NaN,0,NaN
2,TRAIN_0000003,1,1,0,0,1,0,0,1.44,1,1,2,0.0,0.5,0.0,NaN,NaN,1,0.0


## 3. train/test를 서로 독립적으로 변환

`pd.concat([train, test])` 후 통계를 계산하지 않습니다. 같은 함수를 각각 호출해야 실제 평가 규칙을 지킬 수 있습니다.

In [3]:
train_features = build_features(train)
test_features = build_features(test)

train_check = validate_feature_block(train, train_features)
test_check = validate_feature_block(test, test_features)

pd.DataFrame([train_check, test_check], index=['train', 'test'])

,rows,features,row_id_unique,null_cells,infinite_cells,all_numeric,prefix_ok,uses_target,uses_test_aggregate
train,20000,25,True,0,0,True,True,False,False
test,5,25,True,0,0,True,True,False,False


## 4. 테스트 독립성 검사

행을 셔플하거나 절반만 남겨도 동일 `row_id`의 피처가 정확히 같아야 합니다. 이 검사는 test 내부 빈도, rolling, 행 순서 의존성을 탐지합니다.

In [4]:
assert_row_independence(train)
assert_row_independence(test)
print('PASS: shuffle/subset 이후에도 row_id별 피처가 동일합니다.')

PASS: shuffle/subset 이후에도 row_id별 피처가 동일합니다.


## 5. 생성 피처 검토

모델 팀이 빠르게 리뷰할 수 있도록 값 예시와 dtype, 결측률, 범위를 함께 확인합니다.

In [5]:
train_features.head(5)

,row_id,demo__balls_norm,demo__strikes_norm,demo__count_diff,demo__is_full_count,demo__is_two_strike,demo__abs_score_diff,demo__is_close_game,demo__is_trailing,demo__is_late_inning,...,demo__pitcher_history_log1p,demo__batter_history_log1p,demo__pitcher_history_confidence,demo__pitcher_rate_missing,demo__batter_rate_missing,demo__pitcher_success_shrunk,demo__batter_success_shrunk,demo__recent_pitcher_success,demo__recent_career_success_gap,demo__historical_control_risk
0,TRAIN_0000001,0.000000,0.0,0.0,0,0,0.0,1,0,0,...,0.000000,0.000000,0.000000,1,1,0.500000,0.500000,0.500000,0.000000,0.000000
1,TRAIN_0000002,0.000000,0.0,0.0,0,0,0.0,1,0,0,...,0.693147,0.000000,0.004975,0,1,0.497512,0.500000,0.199005,-0.497512,1.000000
2,TRAIN_0000003,0.333333,0.0,1.0,0,0,0.0,1,0,0,...,1.098612,0.693147,0.009901,0,0,0.495050,0.498339,0.198020,-0.495050,0.500000
3,TRAIN_0000004,0.000000,0.0,0.0,0,0,0.0,1,0,0,...,1.386294,0.000000,0.014778,0,1,0.492611,0.500000,0.197044,-0.492611,0.666667
4,TRAIN_0000005,0.000000,0.5,-1.0,0,0,0.0,1,0,0,...,1.609438,0.693147,0.019608,0,0,0.495098,0.501661,0.348039,-0.245098,0.500000


In [6]:
summary = feature_summary(train_features)
summary

,feature,dtype,missing_n,missing_pct,nunique,min,median,max
0,demo__balls_norm,float32,0,0.0,4,0.000000,0.333333,1.000000
1,demo__strikes_norm,float32,0,0.0,3,0.000000,0.500000,1.000000
2,demo__count_diff,float32,0,0.0,6,-2.000000,0.000000,3.000000
3,demo__is_full_count,int8,0,0.0,2,0.000000,0.000000,1.000000
4,demo__is_two_strike,int8,0,0.0,2,0.000000,0.000000,1.000000
5,demo__abs_score_diff,float32,0,0.0,18,0.000000,2.000000,19.000000
6,demo__is_close_game,int8,0,0.0,2,0.000000,0.000000,1.000000
7,demo__is_trailing,int8,0,0.0,2,0.000000,0.000000,1.000000
8,demo__is_late_inning,int8,0,0.0,2,0.000000,0.000000,1.000000
9,demo__late_close_game,int8,0,0.0,2,0.000000,0.000000,1.000000


## 6. 여러 팀원의 피처 묶음 조립

모델 담당자는 각 모듈의 반환값을 아래처럼 `row_id` 기준 one-to-one으로 합칩니다. prefix가 겹치면 조립 함수가 즉시 실패합니다.

In [7]:
base_ids = train[[ID_COLUMN]].copy()

# 실제 사용 예시:
# blocks = [
#     feat_jisu_count.build_features(train),
#     feat_minsu_history.build_features(train),
#     feat_yuna_trackman.build_features(train),
# ]
blocks = [train_features]
assembled = merge_feature_blocks(base_ids, blocks)
print('assembled:', assembled.shape)
assembled.head(3)

assembled: (20000, 26)


,row_id,demo__balls_norm,demo__strikes_norm,demo__count_diff,demo__is_full_count,demo__is_two_strike,demo__abs_score_diff,demo__is_close_game,demo__is_trailing,demo__is_late_inning,...,demo__pitcher_history_log1p,demo__batter_history_log1p,demo__pitcher_history_confidence,demo__pitcher_rate_missing,demo__batter_rate_missing,demo__pitcher_success_shrunk,demo__batter_success_shrunk,demo__recent_pitcher_success,demo__recent_career_success_gap,demo__historical_control_risk
0,TRAIN_0000001,0.000000,0.0,0.0,0,0,0.0,1,0,0,...,0.000000,0.000000,0.000000,1,1,0.500000,0.500000,0.500000,0.000000,0.0
1,TRAIN_0000002,0.000000,0.0,0.0,0,0,0.0,1,0,0,...,0.693147,0.000000,0.004975,0,1,0.497512,0.500000,0.199005,-0.497512,1.0
2,TRAIN_0000003,0.333333,0.0,1.0,0,0,0.0,1,0,0,...,1.098612,0.693147,0.009901,0,0,0.495050,0.498339,0.198020,-0.495050,0.5


## 7. 제출 전 스모크 산출물 저장

공유의 기준은 코드와 manifest입니다. 대용량 전체 피처 파일은 필요할 때 동일 코드로 재생성합니다.

In [8]:
NOTEBOOK_OUT = HERE / 'notebook_outputs'
NOTEBOOK_OUT.mkdir(exist_ok=True)

train_sample_path = NOTEBOOK_OUT / 'demo_train_features_20000.parquet'
test_sample_path = NOTEBOOK_OUT / 'demo_test_features.parquet'
summary_path = NOTEBOOK_OUT / 'demo_feature_summary.csv'

train_features.to_parquet(train_sample_path, index=False)
test_features.to_parquet(test_sample_path, index=False)
summary.to_csv(summary_path, index=False, encoding='utf-8-sig')

print(train_sample_path)
print(test_sample_path)
print(summary_path)

C:\Users\isj67\Desktop\LGAIMERS\experiment\control_success_feature_template\notebook_outputs\demo_train_features_20000.parquet
C:\Users\isj67\Desktop\LGAIMERS\experiment\control_success_feature_template\notebook_outputs\demo_test_features.parquet
C:\Users\isj67\Desktop\LGAIMERS\experiment\control_success_feature_template\notebook_outputs\demo_feature_summary.csv


## 팀원 제출 체크리스트

1. 파일명: `feat_<이름>_<주제>.py`
2. `SPEC.owner`, `SPEC.name`, `SPEC.version`, `SPEC.prefix` 수정
3. prefix 예: `jisu_count__`, `minsu_hist__`
4. `required_columns`와 `feature_columns`를 빠짐없이 명시
5. `build_features()`는 원본을 수정하지 않고 `row_id + 새 피처`만 반환
6. target, row_id 숫자, test 내부 집계 사용 금지
7. 스모크 테스트·행 독립성 검사를 통과한 manifest와 함께 제출
8. 피처 설명과 예상 가설을 README/PR에 3~5줄로 기록
9. target encoding이나 학습형 임베딩은 별도 stateful 모듈로 만들고 CV fold 내부에서만 fit
